# S3.18 - orderBy, sort
**Date completed:** October 2026  
**Status:**  Completed\
**Dataset:** samples.tpch.orders, 7.5M rows\
**Key concept:** orderBy is a wide transformation (shuffle). Reduce the data first, then sort.

In [0]:
# ============================================================
# S3.18 — Cell 2
# orderBy(), sort(), NULL Ordering and Partition Sorting
#
# Goal:
# 1. Use orderBy() ascending and descending.
# 2. Understand sort() as an alias of orderBy().
# 3. Sort by multiple columns.
# 4. Control NULL placement explicitly.
# 5. Understand deterministic ordering and tie-breakers.
# 6. Build the correct Top-N pattern.
# 7. Understand why orderBy() commonly requires a shuffle.
# 8. Compare global orderBy() with sortWithinPartitions().
#
# Dataset:
# Controlled sample data
# samples.tpch.orders
#
# Environment:
# Databricks Serverless
#
# Important:
# A DataFrame has no guaranteed business row order unless
# an explicit ordering is applied.
# ============================================================

import pyspark.sql.functions as F


print("=== S3.18: orderBy() AND sort() ===")
print()


# ------------------------------------------------------------
# Step 1 — Controlled Data with NULLs
# ------------------------------------------------------------

data = [
    (1, "Delhi",  2500.0),
    (2, "Mumbai", None),
    (3, "Pune",   1800.0),
    (4, "Delhi",  3200.0),
    (5, "Mumbai", 950.0),
    (6, "Pune",   None),
]


df = spark.createDataFrame(
    data,
    [
        "order_id",
        "city",
        "amount"
    ]
)


print("=== STEP 1: SOURCE DATA ===")

df.show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 2 — Basic Ascending and Descending Sort
#
# orderBy("amount")
# → ascending by default
#
# desc()
# → descending
# ------------------------------------------------------------

print("=== STEP 2: ASCENDING / DESCENDING ===")


print("\nAscending:")

df.orderBy(F.col("amount").asc()).show(truncate=False)

print("\nDescending:")

df.orderBy(F.col("amount").desc()).show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 3 — sort() Is an Alias of orderBy()
#
# These have the same sorting semantics:
#
# df.orderBy(...)
# df.sort(...)
#
# IMPORTANT:
#
# If multiple rows have equal sort-key values, their relative
# order is not guaranteed unless another tie-breaker is added.
#
# Our data contains two NULL amounts.
#
# Therefore comparing:
#
# orderBy("amount")
#
# directly with:
#
# sort("amount")
#
# using Python list equality is not a strong validation because
# the two NULL rows are tied.
#
# We add order_id as a deterministic tie-breaker.
# ------------------------------------------------------------

print(
    "=== STEP 3: orderBy() vs sort() ==="
)


ordered_result = (

    df

    .orderBy(
        F.col("amount").asc_nulls_last(),
        F.col("order_id").asc()).collect())

sorted_result = (

    df

    .sort(
        F.col("amount").asc_nulls_last(),
        F.col("order_id").asc())

    .collect()
)


same_result = (ordered_result == sorted_result)

print(
    f"orderBy() and sort() give the same result: "
    f"{same_result}")

assert same_result, ("sort() and orderBy() results differ")

print()

In [0]:
# ------------------------------------------------------------
# Step 4 — Default NULL Placement
#
# Spark default ordering:
#
# ASC
# → NULLS FIRST
#
# DESC
# → NULLS LAST
#
# We can override this explicitly.
# ------------------------------------------------------------

print(
    "=== STEP 4: NULL PLACEMENT ==="
)

print("\nDefault ascending:")

df.orderBy(
    F.col("amount").asc()).show(truncate=False)


print("\nDefault descending:")

df.orderBy(F.col("amount").desc()).show(truncate=False)

print("\nAscending — NULLs FIRST explicitly:")

df.orderBy(F.col("amount").asc_nulls_first()).show(truncate=False)

print("\nAscending — NULLs LAST:")

df.orderBy(F.col("amount").asc_nulls_last()).show(truncate=False)

print("\nDescending — NULLs FIRST:")

df.orderBy(F.col("amount").desc_nulls_first()).show(truncate=False)

print("\nDescending — NULLs LAST explicitly:")

df.orderBy(F.col("amount").desc_nulls_last()).show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 5 — Multi-Column Sort
#
# Requirement:
#
# 1. city ascending
# 2. amount descending
# 3. NULL amount last
# 4. order_id ascending as deterministic tie-breaker
#
#
# Sorting priority works left → right.
#
# city
#   ↓
# amount
#   ↓
# order_id
# ------------------------------------------------------------

print(
    "=== STEP 5: MULTI-COLUMN SORT ==="
)


df_multi_sorted = (

    df

    .orderBy(

        F.col("city").asc(),

        F.col("amount").desc_nulls_last(),

        F.col("order_id").asc()))

df_multi_sorted.show(
    truncate=False
)

print()

In [0]:
# ------------------------------------------------------------
# Step 6 — Why Tie-Breakers Matter
#
# Suppose two customers have exactly the same revenue.
#
# Sorting only by:
#
# total_revenue DESC
#
# does not specify which tied customer comes first.
#
# Add:
#
# customer_id ASC
#
# to make the business ordering deterministic.
# ------------------------------------------------------------

print(
    "=== STEP 6: DETERMINISTIC ORDERING ==="
)


print("""
Example:

orderBy(
    total_revenue DESC,
    customer_id ASC
)

Primary key:
total_revenue

Tie-breaker:
customer_id

If two rows have the same revenue,
customer_id determines their final order.

Important:

A deterministic Top-N needs enough ordering columns
to resolve relevant ties.
""")

In [0]:
# ------------------------------------------------------------
# Step 7 — Load Real TPC-H Orders
# ------------------------------------------------------------

df_orders = spark.table(
    "samples.tpch.orders"
)

print(
    "=== STEP 7: TPC-H ORDERS LOADED ==="
)

print()

In [0]:
# ------------------------------------------------------------
# Step 8 — Correct Top-N Pattern
#
# Business requirement:
#
# Top 10 customers by total revenue.
#
# Correct sequence:
#
# detailed orders
#      ↓
# groupBy(customer)
#      ↓
# sum(revenue)
#      ↓
# orderBy(revenue DESC)
#      ↓
# deterministic tie-breaker
#      ↓
# limit(10)
#
# Target grain:
# one row per customer.
# ------------------------------------------------------------

print(
    "=== STEP 8: TOP 10 CUSTOMERS BY REVENUE ==="
)


df_top = (

    df_orders

    .groupBy(
        "o_custkey")

    .agg(
        F.sum("o_totalprice").alias("total_revenue"))

    .orderBy(
        F.col("total_revenue").desc(),

        F.col("o_custkey").asc()).limit(10))

df_top.show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 9 — orderBy().limit() vs limit().orderBy()
#
# This distinction is critical.
#
#
# CORRECT:
#
# orderBy(...)
#     ↓
# limit(10)
#
# Means:
#
# Sort the required result
# and return the Top 10.
#
#
# WRONG for Top-N:
#
# limit(10)
#     ↓
# orderBy(...)
#
# Means:
#
# Take some limited subset first,
# then sort only those rows.
#
# It does NOT produce the global Top 10.
# ------------------------------------------------------------

print(
    "=== STEP 9: LIMIT PLACEMENT ==="
)


print("""
Correct Top-N:

df
  .orderBy(metric.desc())
  .limit(10)

→ Global Top 10 according to metric.


Not equivalent:

df
  .limit(10)
  .orderBy(metric.desc())

→ Limits first.
→ Sorts only that limited subset.

Therefore:

For business Top-N:
SORT first
LIMIT second.
""")

In [0]:
# ------------------------------------------------------------
# Step 10 — Global Sort Execution Plan
#
# A global orderBy() commonly needs rows redistributed so
# Spark can establish ordering across partitions.
#
# Look in the plan for something similar to:
#
# Exchange
# rangepartitioning(...)
#
# followed by:
#
# Sort
#
# Exact plan details can vary because of:
#
# optimizer decisions
# AQE
# Serverless execution
# source partitioning
# ------------------------------------------------------------

print(
    "=== STEP 10: GLOBAL SORT PLAN ==="
)


df_global_sort = (

    df_orders

    .select(
        "o_orderkey",
        "o_totalprice"
    )

    .orderBy(F.col("o_totalprice").desc()))

df_global_sort.explain(
    "formatted"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 11 — Conceptual Global Sort
# ------------------------------------------------------------

print(
    "=== STEP 11: GLOBAL SORT CONCEPT ==="
)


print("""
Conceptually:

Partition 1 ─┐
Partition 2 ─┼─→ Exchange / Range Partitioning
Partition 3 ─┘
                    ↓
              Sort partitions
                    ↓
             Globally ordered
                 DataFrame


orderBy():

→ establishes DataFrame-wide ordering.

Because records across partitions must be coordinated,
a global sort commonly requires a shuffle/exchange.
""")

In [0]:
# ------------------------------------------------------------
# Step 12 — sortWithinPartitions()
#
# sortWithinPartitions():
#
# Sorts records only INSIDE each existing partition.
#
# It does NOT establish global ordering across partitions.
#
#
# Example:
#
# Partition 1:
# 900
# 700
# 100
#
# Partition 2:
# 1000
# 800
# 200
#
# Each partition is descending,
# but the complete DataFrame is NOT globally sorted.
#
#
# sortWithinPartitions() itself does not require a global
# range-shuffle merely to establish global order.
#
# However, an upstream operation may already have introduced
# a shuffle.
# ------------------------------------------------------------

print(
    "=== STEP 12: sortWithinPartitions() PLAN ==="
)


df_partition_sort = (

    df_orders

    .select(
        "o_orderkey",
        "o_totalprice"
    )
    .sortWithinPartitions(F.col("o_totalprice").desc()))

df_partition_sort.explain(
    "formatted"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 13 — orderBy() vs sortWithinPartitions()
# ------------------------------------------------------------

print(
    "=== STEP 13: GLOBAL vs PARTITION SORT ==="
)


print("""
orderBy():

Partition 1 ─┐
Partition 2 ─┼─→ coordinate ordering across partitions
Partition 3 ─┘
                 ↓
          Global ordering


sortWithinPartitions():

Partition 1
→ sorted internally

Partition 2
→ sorted internally

Partition 3
→ sorted internally

BUT:

Partition 1 + Partition 2 + Partition 3

are not guaranteed to be globally ordered.
""")

In [0]:
# ------------------------------------------------------------
# Step 14 — Row Order Without orderBy()
#
# DataFrames are distributed collections.
#
# Without explicit ordering, do not rely on:
#
# show()
# first()
# head()
# take()
# limit()
#
# returning rows in a business-defined order.
# ------------------------------------------------------------

print(
    "=== STEP 14: NO IMPLICIT ROW ORDER ==="
)


print("""
Do not assume:

df.show(10)

means:

"the first 10 business rows"


Do not assume:

df.limit(10)

means:

"the smallest 10 IDs"


If business ordering matters:

orderBy(...)
must define it explicitly.
""")

In [0]:
# ------------------------------------------------------------
# Step 15 — Transformations and Actions
# ------------------------------------------------------------

print(
    "=== STEP 15: TRANSFORMATIONS vs ACTIONS ==="
)


print("""
Transformations:

orderBy()
sort()
sortWithinPartitions()
limit()
groupBy()
agg()


Actions used:

show()
collect()


Important:

orderBy() builds an ordering transformation.

The actual distributed sort executes when an action
requests the result.
""")

In [0]:
# ------------------------------------------------------------
# Step 16 — Final Summary
# ------------------------------------------------------------

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
orderBy():

→ Global DataFrame ordering.


sort():

→ Alias of orderBy().
→ Same sorting semantics.


ASCENDING:

col.asc()


DESCENDING:

col.desc()


DEFAULT NULL ORDER:

ASC
→ NULLS FIRST

DESC
→ NULLS LAST


EXPLICIT NULL CONTROL:

asc_nulls_first()
asc_nulls_last()
desc_nulls_first()
desc_nulls_last()


MULTI-COLUMN ORDER:

orderBy(
    primary_sort,
    secondary_sort,
    tie_breaker
)

Sorting priority is evaluated left to right.


DETERMINISTIC ORDER:

If primary sort values can tie,
add a meaningful tie-breaker.


TOP-N:

aggregate if required
       ↓
orderBy(metric DESC, tie_breaker)
       ↓
limit(N)


IMPORTANT:

orderBy().limit()
≠
limit().orderBy()


GLOBAL SORT:

orderBy()
commonly requires:

Exchange / range partitioning
        ↓
Sort

PARTITION SORT:

sortWithinPartitions()

→ sorts only within each partition
→ does not guarantee global ordering


CORE RULE:

No explicit orderBy()
→ no guaranteed business ordering.
""")


print("=== S3.18 CELL 2 COMPLETED ===")

# S3.18 — Key Takeaways: orderBy(), sort() and Sorting

## 1. Basics

- `orderBy()` performs DataFrame-wide ordering.
- `sort()` is an alias of `orderBy()` with the same sorting semantics.
- `col.asc()` sorts ascending.
- `col.desc()` sorts descending.
- In multi-column sorting, priority is evaluated from left to right.

Example:

```python
df.orderBy(
    F.col("city").asc(),
    F.col("amount").desc()
)
```

Conceptually:

```text
Sort by city first
        ↓
Within each city, sort by amount
```

---

## 2. NULL Placement

Default Spark ordering:

```text
ASC
→ NULLs first

DESC
→ NULLs last
```

Explicit control:

```python
F.col("amount").asc_nulls_first()
F.col("amount").asc_nulls_last()

F.col("amount").desc_nulls_first()
F.col("amount").desc_nulls_last()
```

Use explicit NULL placement when the business rule depends on where missing values appear.

---

## 3. Deterministic Ordering

If two rows have the same sort-key value, their relative order is not guaranteed unless another sort key resolves the tie.

Example:

```python
df.orderBy(
    F.col("total_revenue").desc(),
    F.col("customer_id").asc()
)
```

Here:

```text
Primary sort
→ total_revenue DESC

Tie-breaker
→ customer_id ASC
```

### Important

This:

```python
df.orderBy("amount")
```

may not fully define the order if several rows have the same `amount`.

For reproducible business ordering, use enough sort keys to resolve relevant ties.

---

## 4. Comparing Sorted Results

If two DataFrames are compared after sorting, the sorting must define the complete required order.

Weak comparison:

```python
df.orderBy("amount").collect()
```

when duplicate `amount` values exist.

Better:

```python
df.orderBy(
    F.col("amount").asc(),
    F.col("order_id").asc()
).collect()
```

The second key resolves ties.

---

## 5. Top-N Pattern

Correct business Top-N pattern:

```text
Detailed data
      ↓
Aggregate to target grain
      ↓
orderBy(metric DESC, tie-breaker)
      ↓
limit(N)
```

Example:

```python
(
    df
    .groupBy("customer_id")
    .agg(
        F.sum("revenue").alias("total_revenue")
    )
    .orderBy(
        F.col("total_revenue").desc(),
        F.col("customer_id").asc()
    )
    .limit(10)
)
```

---

## 6. orderBy().limit() vs limit().orderBy()

Correct:

```python
df.orderBy(
    F.col("metric").desc()
).limit(10)
```

means:

```text
Order the complete result
        ↓
Take Top 10
```

This gives the global Top-N.

---

Not equivalent:

```python
df.limit(10).orderBy(
    F.col("metric").desc()
)
```

means:

```text
Take a limited subset first
        ↓
Sort only that subset
```

Therefore it does **not** mean global Top-N.

### Rule

```text
For Top-N:
SORT first
LIMIT second
```

---

## 7. Global Sort Cost

A global `orderBy()` commonly requires Spark to coordinate ordering across partitions.

Typical conceptual plan:

```text
Input partitions
      ↓
Exchange / Range Partitioning
      ↓
Sort within resulting partitions
      ↓
Globally ordered result
```

Exact execution can vary because of:

```text
Catalyst optimizer
Adaptive Query Execution
Source partitioning
Serverless execution
Physical-plan optimisation
```

Verify using:

```python
df.explain("formatted")
```

and Databricks Query Profile when needed.

---

## 8. sortWithinPartitions()

```python
df.sortWithinPartitions(
    F.col("amount").desc()
)
```

sorts rows only inside each existing partition.

Example:

```text
Partition 1:
900
700
100

Partition 2:
1000
800
200
```

Each partition is sorted internally, but the complete DataFrame is not globally ordered.

Therefore:

```text
sortWithinPartitions()
→ local partition order

orderBy()
→ global DataFrame order
```

### Important

`sortWithinPartitions()` itself does not establish a global range-partitioning shuffle just to create global order.

However, an earlier transformation in the query may already have introduced an Exchange.

---

## 9. No Implicit DataFrame Order

Without explicit ordering, do not treat these as business-defined ordering operations:

```text
show()
first()
head()
take()
limit()
```

Example:

```python
df.limit(10)
```

does not mean:

```text
10 smallest IDs
```

or:

```text
10 highest revenue rows
```

unless ordering was defined first.

### Rule

```text
If business order matters
→ orderBy() must define it
```

---

## 10. sort() vs orderBy()

These are aliases:

```python
df.orderBy(...)
```

and:

```python
df.sort(...)
```

They have the same sorting semantics.

Choose whichever style is clearer for your codebase.

---

## 11. Quick Comparison

| Operation | Scope | Global Order? | Typical Use |
|---|---|---:|---|
| `orderBy()` | Entire DataFrame | Yes | Business/global ordering |
| `sort()` | Entire DataFrame | Yes | Alias of `orderBy()` |
| `sortWithinPartitions()` | Each partition | No | Partition-local ordering |
| `limit()` | Result-size restriction | No | Restrict rows |

---

## 12. Performance Principle

Global sorting is expensive compared with many narrow transformations because Spark may need to redistribute data and coordinate ordering across partitions.

Therefore:

```text
Sort only when the business requirement needs ordering.
```

Do not add `orderBy()` merely for cosmetic reasons in intermediate pipeline steps.

---

## 13. Observed from Notebook Output

### Step 4 — NULL Placement

Default ascending:

```text
NULL rows appear: __________
```

Default descending:

```text
NULL rows appear: __________
```

Expected Spark semantics:

```text
ASC  → NULLs first
DESC → NULLs last
```

---

### Step 10 — Global orderBy() Plan

Number of `Exchange` entries observed:

```text
__________
```

Look for plan details such as:

```text
Exchange
rangepartitioning(...)
Sort
```

---

### Step 12 — sortWithinPartitions() Plan

Number of `Exchange` entries observed:

```text
__________
```

Interpret the actual plan rather than assuming the answer.

If an Exchange appears, check whether it came from an upstream operation rather than from `sortWithinPartitions()` itself.

---

# 14. Related Interview Questions and Answers

### Q1. What is the difference between `sort()` and `orderBy()`?

There is no semantic difference. `sort()` is an alias of `orderBy()`.

---

### Q2. Does `orderBy()` guarantee deterministic results when values tie?

Not completely if the sort keys are tied.

Add a deterministic tie-breaker.

Example:

```python
orderBy(
    F.col("revenue").desc(),
    F.col("customer_id").asc()
)
```

---

### Q3. What is the correct Top-N pattern?

```text
Aggregate if required
      ↓
orderBy(metric DESC, tie-breaker)
      ↓
limit(N)
```

---

### Q4. Why is `limit(10).orderBy(...)` not a global Top 10?

Because Spark first limits the DataFrame and then sorts only that subset.

For global Top-N, sorting must happen before the limit.

---

### Q5. What is the difference between `orderBy()` and `sortWithinPartitions()`?

`orderBy()` establishes global DataFrame ordering.

`sortWithinPartitions()` only sorts rows inside each partition.

---

### Q6. Why can `orderBy()` be expensive?

A global sort commonly requires data redistribution through an Exchange/range-partitioning step before Spark can establish ordering across partitions.

---

### Q7. What is the default NULL ordering?

```text
Ascending
→ NULLs first

Descending
→ NULLs last
```

Explicit NULL-order functions should be used when the requirement must be clear.

---

### Q8. Does `show(10)` return the first 10 business rows?

No.

Without an explicit `orderBy()`, those rows should not be interpreted as business-defined first rows.

---

### Q9. Why should Top-N usually include a tie-breaker?

If several rows have the same primary metric, the tie-breaker defines which row comes first and makes the result reproducible.

---

## 15. Core Understanding

```text
Need global order?
→ orderBy() / sort()

Need ascending?
→ asc()

Need descending?
→ desc()

Need explicit NULL placement?
→ *_nulls_first / *_nulls_last

Need deterministic result?
→ add tie-breaker

Need global Top-N?
→ orderBy() → limit()

Need only partition-local order?
→ sortWithinPartitions()

Need plan evidence?
→ explain("formatted") / Query Profile
```

Core rule:

```text
No explicit ordering
→ no guaranteed business row order
```

**S3.18 — orderBy() and sort() completed after checkpoint validation.**